# Cvičení 1: Neuronové sítě v Keras – Rozpoznávání rukou psaných číslic (MNIST)

> **Cíl cvičení:** Vytvořit a natrénovat první plně propojenou neuronovou síť (Multilayer Perceptron – MLP)
> v knihovně **TensorFlow / Keras** pro rozpoznávání číslic 0 až 9 z datasetu **MNIST** ($28 \times 28$ pixelů).
> Normalizovat data, převést cílovou proměnnou do kategoriálního formátu (`to_categorical`), sestavit architekturu,
> natrénovat model s optimalizátorem **Adam** a ztrátovou funkcí **Cross-Entropy**, vyhodnotit přesnost na testovací sadě
> a vizualizovat predikci pro vybraný obrázek pomocí `plt.imshow()`.

---

## 1. Import knihoven a načtení datasetu MNIST

Importujeme potřebné třídy z `tensorflow.keras` a `matplotlib.pyplot`.
Dataset MNIST načteme přímo z modulu `keras.datasets`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.utils import to_categorical

# Načtení datasetu MNIST
mnist_dataset = keras.datasets.mnist
(X_train, y_train), (X_test, y_test) = mnist_dataset.load_data()

print(f"Rozměry trénovací sady X: {X_train.shape}, štítky y: {y_train.shape}")
print(f"Rozměry testovací sady  X: {X_test.shape},  štítky y: {y_test.shape}")

## 2. Náhled na data a vizualizace prvního vzorku

Zobrazíme si vzorový obrázek ze sady pomocí `plt.imshow()` s černobílou colormapou.

In [ ]:
plt.figure(figsize=(4, 4))
plt.imshow(X_train[0], cmap="gray")
plt.title(f"Skutečná číslice (Label): {y_train[0]}")
plt.axis("off")
plt.show()

## 3. Předzpracování dat (Normalizace a Kategoriální kódování)

- **Nezávislá proměnná (obrázky $X$):** Převedeme hodnoty pixelů z rozsahu $[0, 255]$ do intervalu $[0.0, 1.0]$.
- **Závislá proměnná (štítky $y$):** Převedeme celočíselné labely na One-Hot vektory o délce 10 pomocí `to_categorical()`.

In [ ]:
# 1. Normalizace pixelů
X_train_norm = X_train.astype("float32") / 255.0
X_test_norm = X_test.astype("float32") / 255.0

# 2. Převod cílové proměnné na One-Hot (kategoriální)
y_train_cat = to_categorical(y_train, num_classes=10)
y_test_cat = to_categorical(y_test, num_classes=10)

print(f"Původní štítek prvního vzorku: {y_train[0]}")
print(f"One-Hot zakódovaný vektor:     {y_train_cat[0]}")

## 4. Definice architektury neuronové sítě

Sestavíme model `Sequential` skládající se z:
1. **Vstupní vrstvy:** `Flatten(input_shape=(28, 28))` – zploští matici pixelů na vektor 784 hodnot.
2. **Skryté vrstvy:** `Dense(128, activation='relu')` – plně propojená vrstva se 128 neurony a aktivací ReLU.
3. **Výstupní vrstvy:** `Dense(10, activation='softmax')` – 10 neuronů pro 10 tříd číslic s normalizovanou distribucí pravděpodobností.

In [ ]:
model = Sequential([
    Flatten(input_shape=(28, 28), name="vstupni_zplosteni"),
    Dense(128, activation="relu", name="skryta_vrstva_relu"),
    Dense(10, activation="softmax", name="vystupni_vrstva_softmax")
])

model.summary()

## 5. Kompilace modelu

Použijeme optimalizátor vah **Adam**, ztrátovou funkci **`categorical_crossentropy`** a metriku **`accuracy`**.

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
print("Model byl úspěšně zkompilován!")

## 6. Trénování sítě (`model.fit`)

Zvolíme:
- `epochs=15`: počet průchodů celou trénovací sadou,
- `batch_size=256`: velikost dávky pro jeden krok gradientního sestupu,
- `validation_split=0.15`: 15 % trénovacích dat vyčleníme pro průběžnou validaci.

In [ ]:
history = model.fit(
    X_train_norm,
    y_train_cat,
    epochs=15,
    batch_size=256,
    validation_split=0.15,
    verbose=1
)

## 7. Vyhodnocení na testovací sadě a predikce

Otestujeme přesnost sítě na 10 000 nezávislých testovacích obrázcích a spočteme matici pravděpodobností `y_pred`.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test_norm, y_test_cat, verbose=0)
print(f"Testovací ztráta (Loss):      {test_loss:.4f}")
print(f"Testovací přesnost (Accuracy): {test_accuracy*100:.2f} %")

# Predikce na testovací sadě
y_pred_probs = model.predict(X_test_norm)
y_pred_labels = np.argmax(y_pred_probs, axis=1)

## 8. Inspekce konkrétního vybraného obrázku

Vybereme vzorek z testovací sady (např. index 42), zkontrolujeme skutečnou hodnotu, predikci modelu a zobrazíme obrázek.

In [ ]:
sample_idx = 42
actual_label = y_test[sample_idx]
predicted_label = y_pred_labels[sample_idx]
confidence = y_pred_probs[sample_idx][predicted_label] * 100

print(f"Index testovacího vzorku:      {sample_idx}")
print(f"Skutečná hodnota (Label):     {actual_label}")
print(f"Predikce modelu:              {predicted_label} (Jistota: {confidence:.2f} %)")

plt.figure(figsize=(4, 4))
plt.imshow(X_test[sample_idx], cmap="gray")
plt.title(f"Predikce: {predicted_label} (Skutečnost: {actual_label})")
plt.axis("off")
plt.show()